In [13]:
#necessary imports and loading data

import pandas as pd
import numpy as np
import plotly.graph_objects as GO
import plotly.express as px
from scipy.stats import pearsonr
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly

rainfall_data = pd.read_csv("data.csv")
print(rainfall_data.head())

  REGION  YEAR   JAN   FEB   MAR   APR   MAY    JUN    JUL    AUG    SEP  \
0  INDIA  1901  34.7  37.7  18.0  39.3  50.8  113.4  242.2  272.9  124.4   
1  INDIA  1902   7.4   4.3  19.0  43.5  48.3  108.8  284.0  199.7  201.5   
2  INDIA  1903  17.0   8.3  31.3  17.1  59.5  118.3  297.0  270.4  199.1   
3  INDIA  1904  14.4   9.6  31.8  33.1  72.4  164.8  261.0  206.4  129.6   
4  INDIA  1905  25.3  20.9  42.7  33.7  55.7   93.3  252.8  200.8  178.4   

     OCT   NOV   DEC  ANNUAL  Jan-Feb  Mar-May  Jun-Sep  Oct-Dec  
0   52.7  38.0   8.3  1032.3     72.4    108.1    752.8     99.0  
1   61.5  27.9  24.4  1030.2     11.7    110.8    794.0    113.8  
2  117.9  36.9  17.7  1190.5     25.3    107.9    884.8    172.5  
3   69.0  11.2  16.3  1019.8     24.0    137.4    761.8     96.6  
4   51.4   9.7  10.5   975.3     46.2    132.2    725.4     71.6  


In [14]:
#annual rainfall trands

annual_rain = rainfall_data[['YEAR', 'ANNUAL']]

annual_fig = GO.Figure()
annual_fig.add_trace(GO.Scatter(
    x = annual_rain['YEAR'],
    y = annual_rain['ANNUAL'],
    mode = 'lines',
    name = 'Annual Rain',
    line = dict(color = 'blue', width = 3),
    opacity = 0.8
))

annual_fig.add_trace(GO.Scatter(
    x = annual_rain['YEAR'],
    y = [annual_rain['ANNUAL'].mean()] * len(annual_rain),
    mode = 'lines',
    name = 'Mean Rain',
    line = dict(color='purple', dash = 'dash')
))

annual_fig.update_layout(
    title = 'Annual Rain in INDIA',
    xaxis_title = 'Year',
    yaxis_title = 'Rain(mm)',
    template = 'plotly_white',
    legend = dict(title="Legend"),
    height = 500
)

annual_fig.show()

In [15]:
#finding months with highest and lowest rainfall on average
months = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']
months_avg = rainfall_data[months].mean()

highest_rain_month = months_avg.idxmax()
lowest_rain_month = months_avg.idxmin()

months_fig = px.bar(
    x = months_avg.index,
    y = months_avg.values,
    labels = {
        'x' : 'MONTH', 'y': 'RAIN (mm)'
    },
    title = 'Average monthly rain in India',
    text = months_avg.values
)

months_fig.add_hline(
    y = months_avg.mean(),
    line_dash = 'dash',
    line_color = 'red',
    annotation_text= 'Mean Rain',
    annotation_position = 'top right'
)

months_fig.update_traces(marker_color='green', marker_line_color = 'black', marker_line_width = 1)
months_fig.update_layout(template = 'plotly_white', height = 500)
months_fig.show()

In [16]:
#seasonal rain distribution

seasons = ['Jan-Feb', 'Mar-May', 'Jun-Sep', 'Oct-Dec']
seasons_avg = rainfall_data[seasons].mean()

seasons_fig = px.bar(
    x = seasons_avg.index,
    y = seasons_avg.values,
    labels = {
        'x' : 'Season', 'y' : 'Rain (mm)'
    },
    title = 'Seasonal rain distribution in India',
    text = seasons_avg.values,
    color = seasons_avg.values,
    color_continuous_scale = ['gold', 'skyblue', 'green', 'orange']        
)

seasons_fig.update_traces(marker_line_color = 'orange', marker_line_width = 1)
seasons_fig.update_layout(
    template = 'plotly_white',
    height = 500,
    coloraxis_colorbar = dict(title = 'mm')
)

seasons_fig.update_traces(marker_line_color = 'black', marker_line_width = 1)
seasons_fig.update_layout(template = 'plotly_white', height = 500)
seasons_fig.show()

In [24]:
#calculate rolling average to evaluate impact of climate change

rainfall_data['10-Year Rolling Avg'] = rainfall_data['ANNUAL'].rolling(window=10).mean()

climate_fig = GO.Figure()

climate_fig.add_trace(GO.Scatter(
    x = rainfall_data['YEAR'],
    y = rainfall_data['ANNUAL'],
    mode = 'lines',
    name = 'Annual Rain',
    line = dict(color = 'orange', width = 2),
    opacity = 0.6
))

climate_fig.add_trace(GO.Scatter(
    x = rainfall_data['YEAR'],
    y = rainfall_data['10-Year Rolling Avg'],
    mode = 'lines',
    name = '10-year Rolling Avg',
    line = dict(color = 'green', width = 3)
))

climate_fig.update_layout(
    title = 'Impact of Climate Change on Rain',
    xaxis_title = 'Year',
    yaxis_title = 'Rainfall (mm)',
    template = 'plotly_white',
    legend = dict(title ='Legend'),
    height = 500
)

climate_fig.show()

In [18]:
#identigying drought years
#identifying heavy rain years

mean_rain = rainfall_data['ANNUAL'].mean()
std_dev_rain = rainfall_data['ANNUAL'].std()

drought_years = rainfall_data[rainfall_data['ANNUAL'] < (mean_rain - 1.5 * std_dev_rain)]
extreme_years = rainfall_data[rainfall_data['ANNUAL'] < (mean_rain + 1.5 * std_dev_rain)]

seasons_correlation = {
    season : pearsonr(rainfall_data[season], rainfall_data['ANNUAL'])[0] for season in seasons
}

drought_years_summary = drought_years[['YEAR', 'ANNUAL']].reset_index(drop = True)
extreme_years_summary = extreme_years[['YEAR', 'ANNUAL']].reset_index(drop = True)
seasons_correlation_summary = pd.DataFrame.from_dict(seasons_correlation, orient = 'index', columns=['Correlation'])

print(f'Drought years:\n {drought_years_summary}\n')

print(f'Years with extreme rain:\n {extreme_years_summary}\n')

print('Correlations:')
seasons_correlation_summary

Drought years:
    YEAR  ANNUAL
0  1905   975.3
1  1965   938.4
2  1972   948.5
3  2002   920.8
4  2009   959.3

Years with extreme rain:
      YEAR  ANNUAL
0    1901  1032.3
1    1902  1030.2
2    1903  1190.5
3    1904  1019.8
4    1905   975.3
..    ...     ...
103  2011  1110.1
104  2012  1073.5
105  2013  1216.2
106  2014  1033.7
107  2015  1093.2

[108 rows x 2 columns]

Correlations:


,Correlation
Jan-Feb,0.228913
Mar-May,0.313057
Jun-Sep,0.930027
Oct-Dec,0.531648


In [25]:
#searching for annual anomalies

isolation_forest = IsolationForest(contamination = 0.05, random_state = 42)
rainfall_data['Annual_Anomaly'] = isolation_forest.fit_predict(rainfall_data[['ANNUAL']])

annual_anomalies = rainfall_data[rainfall_data['Annual_Anomaly'] == -1]

months_data = rainfall_data[['JAN', 'FEB', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']]
Monthly_anomalies = isolation_forest.fit_predict(months_data)

rainfall_data['Monthly_Anomaly'] = Monthly_anomalies
Monthly_anomalies_df = rainfall_data[rainfall_data['Monthly_Anomaly'] == -1] [['YEAR'] + months]

annual_anomalies_fig = GO.Figure()

annual_anomalies_fig.add_trace(GO.Scatter(
    x = rainfall_data['YEAR'],
    y = rainfall_data['ANNUAL'],
    mode = 'lines',
    name = 'Annual Rain',
    line = dict(color = 'blue', width = 2),
    opacity = 0.7
))

annual_anomalies_fig.add_trace(GO.Scatter(
    x = annual_anomalies['YEAR'],
    y = annual_anomalies['ANNUAL'],
    mode = 'markers',
    name = 'Anomalous Years',
    marker = dict(color = 'red', size = 9, symbol = 'square')
))

annual_anomalies_fig.add_hline(
    y = rainfall_data['ANNUAL'].mean(),
    line_dash = 'dash',
    line_color = 'purple',
    annotation_text = 'Mean Rain',
    annotation_position = 'bottom right'
)

annual_anomalies_fig.update_layout(
    title = 'Annual rain anomalies',
    xaxis_title = 'Year',
    yaxis_title = 'Rain (mm)',
    template = 'plotly_white',
    legend = dict(title = "Legend"),
    height = 500
)

annual_anomalies_fig.show()

In [26]:
#searching for monthly anomalies

monthly_anomalies = []
for column in months:
    for _, rows in Monthly_anomalies_df.iterrows():
        monthly_anomalies.append({'Year': rows['YEAR'], 'Month': column, 'Rainfall': rows[column]})

Monthly_anomalies_df_long = pd.DataFrame(monthly_anomalies)

monthly_anomalies_fig = px.line(
    rainfall_data,
    x = 'YEAR',
    y = months,
    labels = {'YEAR' : 'Year', 'value' : 'Rainfall (mm)', 'variable' : 'Month'},
    title = 'Monthly rain anomalies',
    color_discrete_sequence = px.colors.qualitative.Safe_r
)

monthly_anomalies_fig.add_trace(GO.Scatter(
    x = Monthly_anomalies_df_long['Year'],
    y = Monthly_anomalies_df_long['Rainfall'],
    mode = 'markers',
    name = 'Anamalous Months',
    marker = dict(color = 'black', size = 6, symbol = 'square')
))

monthly_anomalies_fig.update_layout(
    template = 'plotly_white',
    legend = dict(title="Legend"),
    height = 500
)

In [21]:
#correlation b/w Indian monsoon season and other seasons

monsoon = 'Jun-Sep'
season_cols = ['Jan-Feb', 'Mar-May', 'Oct-Dec']

relations = {}

for season in season_cols:
    if season != monsoon:
        corr, _ = pearsonr(rainfall_data[monsoon], rainfall_data[season])
        relations[season] = corr

correlation_data = pd.DataFrame({
    'Season' : list(relations.keys()),
    'Correlation Coefficient' : list(relations.values()) 
})

correlation_fig = px.bar(
    correlation_data, 
    x = 'Season',
    y = 'Correlation Coefficient',
    title = 'correlation b/w Indian monsoon season and other seasons',
    labels = {'Season' : 'Season', 'Correlation Coefficient' : 'Correlation Coefficient'},
    text = 'Correlation Coefficient',
    color = 'Correlation Coefficient',
    color_continuous_scale = 'agsunset' 
)

correlation_fig.add_hline(
    y = 0,
    line_dash = 'dash',
    line_color = 'red',
    annotation_text= 'No Correaltion',
    annotation_position = 'bottom left'
)

correlation_fig.update_traces(marker_line_color = 'black', marker_line_width = 1)
correlation_fig.update_layout(
    template = 'plotly_white',
    height = 500
) 

In [28]:
#clustering years based on rain patterns

rain_features = rainfall_data[['Jan-Feb', 'Mar-May', 'Jun-Sep', 'Oct-Dec', 'ANNUAL']]
scale = StandardScaler()
scale_features = scale.fit_transform(rain_features)

kmeans = KMeans(n_clusters= 3, random_state = 42)
rainfall_data['Rainfall_Cluster'] = kmeans.fit_predict(scale_features)

cluster_labels = {0: 'Dry', 1: 'Normal', 2: 'Wet'}
rainfall_data['Rainfall_Category'] = rainfall_data['Rainfall_Cluster'].map(cluster_labels)

kmeans_fig = px.scatter(
    rainfall_data,
    x = 'YEAR',
    y = 'ANNUAL',
    color = 'Rainfall_Category',
    title = 'Clustering years based on rain patterns',
    labels = {'YEAR': 'Year', 'ANNUAL': 'Annual Rain (mm)', 'Rainfall_Category': 'Rainfall_Category'},
    color_discrete_sequence = px.colors.qualitative.Set2,
    hover_data = {'Rainfall_Cluster': True, 'Rainfall_Category': True}
)

kmeans_fig.update_layout(
    template = 'plotly_white',
    legend = dict(title='Rain Category'),
    height = 500
)

kmeans_fig.show()

In [23]:
#forecasting future rains

rainfall_data['DATE'] = pd.to_datetime(rainfall_data['YEAR'], format = '%Y')
annual_rainfall_ts = rainfall_data.set_index('DATE')['ANNUAL']

prophet_data = annual_rainfall_ts.reset_index()
prophet_data.columns = ['ds', 'y']

prophet_model = Prophet()
prophet_model.fit(prophet_data)

future = prophet_model.make_future_dataframe(periods=20, freq='YE')
forecast = prophet_model.predict(future)

forecast_fig = plot_plotly(prophet_model, forecast)

forecast_fig.update_layout(
    title = 'Annual Rainfall Forecast using Prophet for next 20yrs',
    xaxis_title = 'Year',
    yaxis_title = 'Rainfall (mm)',
    template = 'plotly_white',
    height = 500
)

forecast_fig.show()

15:13:32 - cmdstanpy - INFO - Chain [1] start processing
15:13:32 - cmdstanpy - INFO - Chain [1] done processing
